# RAG (Retrieval-Augmented Generation) - Production Implementation

**Complete guide with real HuggingFace libraries and production patterns.**

This notebook uses:
- Real models from HuggingFace Hub
- Production-grade patterns
- Error handling and optimization
- Real-world use cases

See also: `llm/concepts/rag.md` for theory and interview Q&A

## Setup

In [ ]:
# Install required packages
# !pip install transformers torch sentence-transformers datasets peft bitsandbytes

import warnings
warnings.filterwarnings('ignore')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Quick Start

In [ ]:
# Simple RAG with HuggingFace
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch

# Load components
retriever = SentenceTransformer('all-MiniLM-L6-v2')
llm = pipeline("text-generation", model="gpt2")

# Documents
docs = [
    "Paris is the capital of France",
    "Tokyo is the capital of Japan",
    "Berlin is the capital of Germany"
]

# Embed documents
doc_embeddings = retriever.encode(docs, convert_to_tensor=True)

# Query
query = "What is the capital of France?"
query_embedding = retriever.encode(query, convert_to_tensor=True)

# Retrieve
hits = util.semantic_search(query_embedding, doc_embeddings, top_k=2)
retrieved_docs = [docs[hit['corpus_id']] for hit in hits[0]]

# Generate
context = " ".join(retrieved_docs)
prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

# result = llm(prompt, max_length=100)
print(f"Retrieved: {retrieved_docs}")

## Production Implementation

In [ ]:
# Production RAG Pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from transformers import pipeline
import torch

class RAGPipeline:
    """Production RAG with dense + sparse and re-ranking"""

    def __init__(self):
        # Components
        self.retriever = SentenceTransformer('all-mpnet-base-v2')
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        self.generator = pipeline("text2text-generation", model="google/flan-t5-base")

        self.documents = []
        self.embeddings = None

    def index_documents(self, docs):
        """Index documents"""
        self.documents = docs
        print(f"Indexing {len(docs)} documents...")
        self.embeddings = self.retriever.encode(
            docs,
            batch_size=32,
            convert_to_tensor=True,
            show_progress_bar=True
        )

    def retrieve_and_rerank(self, query, top_k=10, rerank_k=3):
        """Retrieve then re-rank"""
        # Dense retrieval
        query_emb = self.retriever.encode(query, convert_to_tensor=True)
        hits = util.semantic_search(query_emb, self.embeddings, top_k=top_k)

        # Re-rank
        retrieved_docs = [self.documents[hit['corpus_id']] for hit in hits[0]]
        pairs = [[query, doc] for doc in retrieved_docs]
        scores = self.reranker.predict(pairs)

        # Get top after re-ranking
        reranked_indices = scores.argsort()[::-1][:rerank_k]
        return [retrieved_docs[i] for i in reranked_indices]

    def answer(self, query):
        """End-to-end QA"""
        docs = self.retrieve_and_rerank(query)
        context = " ".join(docs)
        prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

        # result = self.generator(prompt, max_length=100)
        # return result[0]['generated_text']
        return f"Retrieved from: {docs}"

# Usage
rag = RAGPipeline()
docs = ["Fact 1", "Fact 2", "Fact 3"]
rag.index_documents(docs)

# answer = rag.answer("Your question?")
# print(answer)

## Real-World: Vectordb

In [ ]:
# Real-World: RAG with Vector Database
from sentence_transformers import SentenceTransformer
from transformers import pipeline

class ProductionRAGService:
    """RAG with vector database backend"""

    def __init__(self):
        self.retriever = SentenceTransformer('all-MiniLM-L6-v2')
        self.generator = pipeline("text-generation", model="gpt2", device=0)

        # In production: use Pinecone, Weaviate, Milvus, etc.
        # from pinecone import Pinecone
        # self.index = Pinecone(api_key="...").Index("rag-index")

    def index_documents(self, documents, batch_size=32):
        """Index documents to vector DB"""
        embeddings = self.retriever.encode(
            documents,
            batch_size=batch_size,
            show_progress_bar=True
        )

        # In production: upsert to vector DB
        # for i, (doc, emb) in enumerate(zip(documents, embeddings)):
        #     self.index.upsert([(str(i), emb.tolist(), {"text": doc})])

        print(f"Indexed {len(documents)} documents")

    def retrieve(self, query, top_k=5):
        """Retrieve from vector DB"""
        query_emb = self.retriever.encode(query)

        # In production: query vector DB
        # results = self.index.query(query_emb.tolist(), top_k=top_k)
        # return [r['metadata']['text'] for r in results]

        return ["Retrieved doc 1", "Retrieved doc 2"]

    def answer_question(self, question):
        """Answer using retrieval + generation"""
        docs = self.retrieve(question)
        context = " ".join(docs)

        prompt = f"Using context: {context}\n\nAnswer: {question}"
        # answer = self.generator(prompt, max_length=100)
        # return answer[0]['generated_text']

        return f"Answer based on: {docs}"

# Production setup
# rag = ProductionRAGService()
# rag.index_documents(large_document_corpus)
# answer = rag.answer_question("What is...?")

## Production Checklist

- [ ] Load models from HuggingFace Hub
- [ ] Set up GPU device handling
- [ ] Implement batch processing
- [ ] Add error handling
- [ ] Optimize for latency
- [ ] Add logging and monitoring
- [ ] Test with production data
- [ ] Create inference service

## Useful Links

- [HuggingFace Models](https://huggingface.co/models)
- [HuggingFace Documentation](https://huggingface.co/docs/transformers)
- [PEFT Library](https://github.com/huggingface/peft)
- [Sentence Transformers](https://www.sbert.net/)